# Varredura de recall por LLM — o que a busca por palavra-chave perde

O protocolo §3 busca 12 termos. Regex não pega paráfrase, e o caso já está documentado no
próprio corpus: os **Termos vinculantes da Booking.com** dizem

> *"sometimes we show different people different designs, phrasings, products, etc.,
> to find out how they react"*

que é reconhecimento explícito de experimentação comportamental — e dá **zero nos doze
termos**. Rode o `01-keyword-sweep.ipynb` e confira: `003-content-terms-html.txt` fecha com
todas as contagens em 0.

Este notebook mede esse buraco. Uma passada independente sobre os 143 documentos,
instruída a **encontrar** qualquer divulgação de experimentação, devolve trechos candidatos;
o cruzamento com o log de palavras-chave diz quais deles a regex teria perdido.

## A regra dura

**Nenhuma saída de modelo entra no dataset codificado.** O modelo devolve *citação verbatim
ou nada* — nunca um código, nunca um nível, nunca um julgamento. Cada citação é conferida
programaticamente contra o texto congelado; o que não for encontrado literalmente é
**descartado automaticamente**, o que elimina alucinação como ameaça: uma alegação que não
se possa citar simplesmente não sobrevive à checagem.

O que sai daqui é uma lista de *ponteiros para trechos*, adjudicada por humano.

## Sequenciamento

Esta passada é **sua**, não do segundo codificador, e roda **em separado** do IRR. Injetar
estes achados na tela dele durante a segunda codificação trocaria a contaminação por outra
— julgamento de máquina em vez de número lembrado.

## Reprodutibilidade

`temperature` **não existe** nos modelos atuais (Opus 5 / 4.8 / 4.7 devolvem 400). E nunca
garantiu saída idêntica, mesmo quando existia. O que sustenta a reprodutibilidade aqui é
mais forte e é o que vai no replication package:

- model ID pinado, sem sufixo de data;
- prompt e schema versionados neste arquivo;
- resposta bruta salva por documento, com o `sha256` do corpo a que se refere;
- **toda citação verificada contra o texto congelado** — a checagem é determinística,
  independente do modelo ter variado.

## 1. Ambiente e credencial

No Colab, guarde a chave em *Secrets* (🔑 na barra lateral) como `ANTHROPIC_API_KEY` — não cole a chave numa célula, o notebook vai para um repo público.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
    for repo in ("experimented-without-consent", "experimented-without-consent-corpus"):
        if not Path(repo).exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            f"https://github.com/mrcsvg/{repo}.git"], check=True)
    INSTRUMENT, CORPUS = Path("experimented-without-consent"), Path("experimented-without-consent-corpus")
    from google.colab import userdata
    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY"))
else:
    INSTRUMENT = Path.cwd().parent
    CORPUS = INSTRUMENT.parent / "experimented-without-consent-corpus"

sys.path.insert(0, str(INSTRUMENT / "analysis"))
import patterns as P
import anthropic

assert os.environ.get("ANTHROPIC_API_KEY"), "defina ANTHROPIC_API_KEY antes de seguir"
client = anthropic.Anthropic()

MODEL = "claude-opus-5"   # pinado; sem sufixo de data
EFFORT = "medium"         # extração, não raciocínio longo — vale varrer low/high depois
MAX_TOKENS = 8000         # teto cobre thinking + resposta (thinking é ligado por padrão)
print(f"modelo {MODEL} · effort {EFFORT}")

## 2. Corpus e log de palavras-chave

Documentos em quarentena (hash divergente) ficam de fora: são gzip não descompactado gravado
como mojibake, e mandá-los ao modelo seria queimar token em lixo — além de arriscar um
"nada encontrado" que pareceria achado.

In [ ]:
import hashlib, json

manifest = P.load_manifest(CORPUS)

kw_path = INSTRUMENT / "analysis" / "keyword-hits.json"
assert kw_path.exists(), "rode 01-keyword-sweep.ipynb primeiro — este notebook cruza com o log dele"
keyword = json.loads(kw_path.read_text(encoding="utf-8"))
KW_POR_ARQUIVO = {d["file"]: d for docs in keyword["servicos"].values() for d in docs}
QUARENTENA = {q["file"] for q in keyword["quarentena"]}

DOCS = []   # (servico, entry, corpo)
for servico, entradas in manifest["services"].items():
    for e in entradas:
        if e["file"] in QUARENTENA:
            continue
        corpo = P.strip_provenance_header((CORPUS / e["file"]).read_text(encoding="utf-8"))
        if hashlib.sha256(corpo.encode("utf-8")).hexdigest() != e["sha256"]:
            continue
        DOCS.append((servico, e, corpo))

chars = sum(len(c) for _, _, c in DOCS)
print(f"documentos a varrer : {len(DOCS)}  (quarentena: {len(QUARENTENA)})")
print(f"caracteres          : {chars:,}")
print(f"estimativa de input  : ~{chars/4/1000:.0f}k tokens  (~US$ {chars/4/1e6*5*0.5:.2f} no Batch, -50%)")

## 3. O prompt

Descreve o **conceito** de experimentação comportamental, não os 12 termos — buscar os
termos é o que a regex já faz. O objetivo aqui é justamente o que ela não alcança.

Três travas no prompt, e a terceira é a que mais importa: sem citação, não há achado.

In [ ]:
SYSTEM = """You are assisting a documentary audit of what online platforms disclose to users \
about behavioural experimentation (A/B tests, randomised trials, and equivalents).

Your ONLY job is to locate passages. You extract evidence; you never evaluate it.

A passage is relevant if it discloses, in any wording, that the service:
- serves different versions, designs, layouts, prices, or messages to different users
  in order to observe or compare how they respond;
- runs experiments, tests, trials, or studies on users or on user-facing surfaces;
- randomises, assigns, or splits users into groups for comparison;
- studies user behaviour or reactions to changes it introduces.

Relevance does NOT depend on any particular vocabulary. A passage that describes the \
practice without ever using the words "experiment" or "test" is exactly what matters most \
here. For example, wording such as "we sometimes show different people different designs to \
see how they react" is squarely relevant.

NOT relevant: free trials or trial subscription periods; software QA, bug testing, or \
security testing; beta or early-access programmes with no comparison between groups; \
personalisation or recommendation described purely as a product feature, with no mention \
of comparing responses.

RULES

1. Quote verbatim. Copy each passage exactly as it appears in the document, character for \
character. Do not paraphrase, summarise, translate, normalise, correct, or truncate mid-word. \
Quote 1-3 sentences: enough that a reader can judge it without the surrounding document.

2. Never assign a code, level, score, rating, or judgement of severity or adequacy. Do not \
say whether the disclosure is sufficient, clear, or binding. A human coder decides all of \
that. The `note` field is for locating the passage in the document, not for evaluating it.

3. If the document contains no relevant passage, return an empty list. An empty list is a \
perfectly good and expected answer, and is much better than a marginal or invented one. \
Never produce a quote that is not literally present in the text you were given."""

SCHEMA = {
    "type": "object",
    "properties": {
        "passages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "quote": {"type": "string",
                              "description": "Verbatim text copied exactly from the document."},
                    "note":  {"type": "string",
                              "description": "Where in the document this sits (section or heading). Locating only, no evaluation."},
                },
                "required": ["quote", "note"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["passages"],
    "additionalProperties": False,
}

PROMPT_SHA = hashlib.sha256((SYSTEM + json.dumps(SCHEMA, sort_keys=True)).encode()).hexdigest()
print(f"prompt+schema sha256: {PROMPT_SHA[:16]}…   (vai no replication package)")

## 4. Montagem do lote

Documentos longos são cortados em janelas com sobreposição, para que nenhum trecho caia exatamente na emenda. O prefixo de instrução é idêntico em todas as requisições e vai com `cache_control` — o lote inteiro lê o mesmo cache.

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

JANELA, SOBREPOSICAO = 120_000, 2_000   # caracteres

def fatiar(texto):
    if len(texto) <= JANELA:
        return [(0, texto)]
    partes, i = [], 0
    while i < len(texto):
        partes.append((i, texto[i:i + JANELA]))
        i += JANELA - SOBREPOSICAO
    return partes

requisicoes, indice = [], {}
for n, (servico, e, corpo) in enumerate(DOCS):
    for k, (offset, trecho) in enumerate(fatiar(corpo)):
        cid = f"d{n:03d}c{k:02d}"
        indice[cid] = {"servico": servico, "file": e["file"], "url": e["url"],
                       "sha256": e["sha256"], "offset": offset}
        requisicoes.append(Request(
            custom_id=cid,
            params=MessageCreateParamsNonStreaming(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=[{"type": "text", "text": SYSTEM,
                         "cache_control": {"type": "ephemeral"}}],
                output_config={"effort": EFFORT,
                               "format": {"type": "json_schema", "schema": SCHEMA}},
                messages=[{"role": "user", "content":
                           f"<document>\n{trecho}\n</document>\n\n"
                           "List every relevant passage in this document, quoted verbatim. "
                           "Return an empty list if there are none."}],
            ),
        ))

print(f"{len(requisicoes)} requisições para {len(DOCS)} documentos "
      f"({len(requisicoes) - len(DOCS)} fatias extras por tamanho)")

In [ ]:
lote = client.messages.batches.create(requests=requisicoes)
print(f"lote {lote.id} · {lote.processing_status}")
print("Batches costumam fechar em menos de 1h (limite 24h).")

In [ ]:
import time

while True:
    lote = client.messages.batches.retrieve(lote.id)
    c = lote.request_counts
    print(f"\r{lote.processing_status}  ok={c.succeeded} erro={c.errored} "
          f"processando={c.processing}   ", end="", flush=True)
    if lote.processing_status == "ended":
        break
    time.sleep(30)
print("\nconcluído")

## 5. Verificação — o passo que torna isso admissível

Cada citação é procurada **literalmente** no corpo congelado. Espaço em branco é normalizado
dos dois lados (o modelo junta quebras de linha), mas nada mais: maiúsculas, pontuação e
palavras têm que bater. Citação com reticências é quebrada nos fragmentos e cada um é
conferido em separado.

O que não for encontrado é descartado e contado. **Esse número vai no artigo** — é a taxa de
citação não verificável, e reportá-la é o que separa uma varredura auditável de um chute.

In [ ]:
import re

def normalizar(s):
    return " ".join(s.split())

def verificar(citacao, corpo):
    """A citação existe literalmente no documento? Devolve (bool, fragmentos_falhos)."""
    alvo = normalizar(corpo)
    fragmentos = [f for f in re.split(r"\s*(?:\.\.\.|…|\[\.\.\.\])\s*", citacao) if len(f.strip()) > 12]
    falhos = [f for f in fragmentos if normalizar(f) not in alvo]
    return (not falhos), falhos

CORPO = {e["file"]: corpo for _, e, corpo in DOCS}

achados, descartados, erros = [], [], []
for r in client.messages.batches.results(lote.id):
    meta = indice[r.custom_id]
    if r.result.type != "succeeded":
        erros.append({"custom_id": r.custom_id, **meta, "tipo": r.result.type})
        continue
    msg = r.result.message
    if msg.stop_reason == "refusal":
        erros.append({"custom_id": r.custom_id, **meta, "tipo": "refusal"})
        continue
    texto = next((b.text for b in msg.content if b.type == "text"), "{}")
    for p in json.loads(texto).get("passages", []):
        ok, falhos = verificar(p["quote"], CORPO[meta["file"]])
        registro = {**meta, "quote": p["quote"], "note": p["note"]}
        (achados if ok else descartados).append(
            registro if ok else {**registro, "fragmentos_nao_encontrados": falhos})

total = len(achados) + len(descartados)
print(f"trechos propostos      : {total}")
print(f"verificados no corpus  : {len(achados)}")
print(f"descartados (não achados): {len(descartados)}"
      f"   → {100*len(descartados)/max(total,1):.1f}% não verificável")
if erros:
    print(f"requisições com erro   : {len(erros)}")

## 6. O achado de método: o que a regex perdeu

Um trecho verificado é **perdido pela regex** quando nenhum dos 12 termos do §3 ocorre dentro dele. É essa contagem que vira a frase de método no artigo.

In [ ]:
def termos_no_trecho(quote):
    counts, _ = P.scan_text(quote)
    return [t for t, c in counts.items() if c]

perdidos, pegos = [], []
for a in achados:
    termos = termos_no_trecho(a["quote"])
    (pegos if termos else perdidos).append({**a, "termos": termos})

print(f"trechos verificados          : {len(achados)}")
print(f"  contêm ≥1 termo do §3      : {len(pegos)}")
print(f"  SEM nenhum termo do §3     : {len(perdidos)}   ← invisíveis à busca por palavra-chave")

servicos_afetados = sorted({p["servico"] for p in perdidos})
print(f"\nserviços com ao menos um trecho invisível à regex: {len(servicos_afetados)}")
for s in servicos_afetados:
    n = sum(1 for p in perdidos if p["servico"] == s)
    print(f"  {s[:52]:54} {n}")

print("\n── amostra ──")
for p in perdidos[:5]:
    print(f"\n{p['servico'][:44]}  ·  {Path(p['file']).name}")
    print(f"  “{normalizar(p['quote'])[:260]}”")
    print(f"  ({p['note'][:100]})")

In [ ]:
import datetime

destino = INSTRUMENT / "analysis" / "llm-recall-sweep.json"
destino.write_text(json.dumps({
    "gerado_por": "notebooks/02-llm-recall-sweep.ipynb",
    "modelo": MODEL,
    "effort": EFFORT,
    "prompt_schema_sha256": PROMPT_SHA,
    "batch_id": lote.id,
    "corpus_frozen_at": manifest["frozen_at"],
    "documentos_varridos": len(DOCS),
    "documentos_em_quarentena": sorted(QUARENTENA),
    "trechos_verificados": achados,
    "trechos_descartados": descartados,
    "erros": erros,
    "perdidos_pela_regex": perdidos,
}, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"gravado: {destino}  ({destino.stat().st_size/1024:.0f} KB)")

## O que disso vai para o artigo

**Método.** *"Uma varredura independente por LLM (modelo pinado, saída restrita a citação
verbatim, cada citação verificada programaticamente contra o corpus com hash) levantou N
trechos em M serviços que o protocolo de busca por palavra-chave do §3 não alcança, por não
conterem nenhum dos doze termos. Prompt, schema e respostas brutas estão no replication
package."*

**Limitações.** A taxa de citação não verificável, medida acima, entra aqui — declarada, não
escondida.

**Uso de IA.** Esta é a frase que o termo de uso de IA precisa: LLM como **auxílio de recall
e checagem adversarial de ausência**, jamais como codificador; toda saída adjudicada por
humano; nenhuma saída de modelo no dataset codificado.

## E a checagem adversarial dos 0/26

O mesmo lote responde à pergunta mais perigosa que um revisor pode fazer sobre um achado de
ausência — *"ou vocês só não procuraram direito?"*. Uma passada instruída a **encontrar**,
que volta vazia num serviço, é evidência negativa que falha de um jeito diferente da regex e
diferente de um humano ancorado. Para reportar isso, cruze `perdidos`/`achados` com as
variáveis que deram 0 de 26.